# Семинар 3. Первые шаги в SQL

На этом занятии вы познакомитесь с PostgreSQL на практике: сначала напишете выражения SQL
в запросах `SELECT` без таблиц, затем создадите свои таблицы с ограничениями и заполните их данными.

**Как устроен ноутбук.** Каждая задача — описание и ячейка, которая начинается с `%%sql`.
Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей.

**База.** PostgreSQL 17 в Docker, демобаза авиаперевозок Postgres Pro (версия 2016 года).
Как поднять базу, открыть ноутбук и подключиться через DBeaver, написано в файле `README.md`
в корне репозитория.

Таблицы демобазы лежат в схеме `bookings`, свои таблицы создаём в схеме `public`.
В запросах пишем имя таблицы вместе со схемой: `bookings.airports`, `public.my_trips`.

**Схема демобазы.** Таблицы, их ключи и связи; подробное описание столбцов — в [`db/schema.md`](../db/schema.md).

![Схема демобазы авиаперевозок](../db/schema.svg)

In [1]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять аэропортов — всё готово к работе.

In [2]:
%%sql
SELECT * FROM bookings.airports LIMIT 5;

,airport_code,airport_name,city,longitude,latitude,timezone
0,MJZ,Мирный,Мирный,114.038928,62.534689,Asia/Yakutsk
1,NBC,Бегишево,Нижнекамск,52.06,55.34,Europe/Moscow
2,NOZ,Спиченково,Новокузнецк,86.8772,53.8114,Asia/Novokuznetsk
3,NAL,Нальчик,Нальчик,43.6366,43.5129,Europe/Moscow
4,OGZ,Беслан,Владикавказ,44.6066,43.2051,Europe/Moscow


Строк: 5


# Часть 1. Выражения SQL в запросах SELECT

**Памятка**

| Что | Как пишется | Пример |
|---|---|---|
| Запрос без таблицы | `SELECT выражение AS имя;` | `SELECT 2 + 2 AS four;` |
| Склейка строк | `\|\|` | `'ABC' \|\| '-' \|\| 'DEF'` |
| Приведение типа | `значение::тип` | `'2016-10-13'::date`, `10.56::int` |
| Текущий момент | `now()` | `now()::timestamptz(0)` — без долей секунды |
| Интервал | `interval '...'` | `interval '2 months'`, `interval '1 minute'` |
| Дата ± дни | `date + число` | `date '2016-10-13' + 60` |
| Остаток от деления | `%` | `7 % 2` |
| Округление вверх / вниз / к ближайшему | `ceil`, `floor`, `round` | `ceil(7 / 3.0)` |
| Проверка на NULL | `IS NULL`, `IS NOT DISTINCT FROM` | `a IS NOT DISTINCT FROM b` |
| «Переменная» | `WITH params AS (SELECT ... AS x)` | см. задачу 4 |

Деление двух целых — целое: `7 / 3 = 2`. Чтобы получить дробь, одно из чисел должно быть дробным: `7 / 3.0`.

### Задача 1. Соберите сообщение конкатенацией

С помощью выражения SQL сформируйте такой текст в столбце **message_text**:

```
Рейс X из DME в LED. По состоянию на Y продано Z билетов
```

- X = `PG0405`
- Y = `now()` с приведением к типу `timestamptz(0)`
- Z = `97`

Части текста соедините конкатенацией.

In [3]:
%%sql
SELECT 'Рейс ' || 'PG0405' || ' из DME в LED. По состоянию на '
       || now()::timestamptz(0) || ' продано ' || 97 || ' билетов' AS message_text;

,message_text
0,Рейс PG0405 из DME в LED. По состоянию на 2026-10-06 22:40:55+03 продано 97 билетов


Строк: 1


**Разбор.** `||` склеивает строку с чем угодно: время и число PostgreSQL сам приведёт к тексту,
если хотя бы один операнд — строка. `timestamptz(0)` отбрасывает доли секунды.

### Задача 2. Проверьте сравнения и операции

Напишите запрос, который вернёт шесть столбцов с результатами вычислений:
- **c_1**. 10 больше 100
- **c_2**. Число 10 равно строке `'10'`. Явное приведение типов не используйте
- **c_3**. Строка `'10'` меньше или равна строке `'100'`
- **c_4**. Строка `'ABC'` равна строке `'abc'`
- **c_5**. Строка `'Moscow'` больше строки `'MOSCOW'`
- **c_6**. Число 10.56, приведённое к целому, не равно числу 10.5

In [4]:
%%sql
SELECT 10 > 100            AS c_1,
       10 = '10'           AS c_2,
       '10' <= '100'       AS c_3,
       'ABC' = 'abc'       AS c_4,
       'Moscow' > 'MOSCOW' AS c_5,
       10.56::int <> 10.5  AS c_6;

,c_1,c_2,c_3,c_4,c_5,c_6
0,false,true,true,false,false,true


Строк: 1


**Разбор.** Ответ: `false, true, true, false, false, true`.

- **c_2**: `'10'` — строковая константа без типа, PostgreSQL приводит её к типу второго операнда, числу.
- **c_3**: сравниваются строки, посимвольно: `'10'` — начало `'100'`, поэтому она меньше.
- **c_4**: сравнение строк чувствительно к регистру.
- **c_5**: результат зависит от правил сортировки (collation) базы. У нас `en_US.utf8`,
  где строчные буквы идут раньше заглавных, поэтому `false`. При сортировке `C`
  сравниваются коды символов и было бы `true`.
- **c_6**: `10.56::int` округляет до 11, а не отбрасывает дробную часть.

### Задача 3. Посчитайте время на подготовку рейсов

**Дано:**
- flights — количество рейсов, которые нужно подготовить к вылету (7)
- time — время подготовки одного рейса в минутах (40)
- crews — количество бригад техников (3)

Каждая бригада готовит один рейс за раз, бригады работают параллельно.

**Задание.** С помощью выражения SQL посчитайте, сколько минут понадобится на подготовку всех рейсов.
Встроенные функции можно использовать, но задача решается и без них.

Название столбца — **complete_time**.

In [5]:
%%sql
SELECT ceil(7 / 3.0) * 40 AS complete_time;

,complete_time
0,120


Строк: 1


**Разбор.** Ответ: 120 минут — три «волны» подготовки по 40 минут.

Ловушка — целочисленное деление: `7 / 3 * 40` даёт 80, а `ceil(7 / 3) * 40` тоже 80,
потому что `7 / 3` уже равно 2 до округления. Нужна дробь: `7 / 3.0`.
Без функций: `(7 + 3 - 1) / 3 * 40` — округление вверх через целочисленное деление.

### Задача 4. Рассчитайте даты через 2 месяца и через 60 дней

Напишите выражение SQL, которое по дате вылета X найдёт тот же день через 2 месяца,
а также дату через 60 дней. Выполните расчёт при X = `'2016-10-13'`.

Попробуйте задать X отдельно, как переменную.

**Названия столбцов:** **dt1**, **dt2**.

In [6]:
%%sql
WITH params AS (
    SELECT date '2016-10-13' AS x
)
SELECT (x + interval '2 months')::date AS dt1,
       x + 60                          AS dt2
FROM params;

,dt1,dt2
0,2016-12-13,2016-12-12


Строк: 1


**Разбор.** Ответ: `2016-12-13` и `2016-12-12` — два месяца и 60 дней не одно и то же.

`date + interval` даёт `timestamp`, поэтому приводим обратно к `date`.
`date + число` прибавляет дни и сразу возвращает `date`.
В SQL нет переменных в привычном смысле: значение задаём один раз в `WITH` и берём оттуда.

### Задача 5. Рассчитайте время окончания подготовки

**Дано:**
- flights — количество рейсов (10)
- time — время подготовки одного рейса в минутах (40)
- crews — количество бригад (3)

**Задание.** С помощью выражения SQL определите, к какому времени будет подготовлен последний рейс,
если начать сейчас. Текущее время — `now()`. Точность — до секунд, часовой пояс не нужен.

Название столбца — **complete_time**.

In [7]:
%%sql
SELECT (now() + ceil(10 / 3.0) * 40 * interval '1 minute')::timestamp(0) AS complete_time;

,complete_time
0,2026-10-07 01:20:55


Строк: 1


**Разбор.** Ответ: текущее время плюс 160 минут (четыре волны по 40).

Число минут превращаем в интервал умножением на `interval '1 minute'`.
`timestamp(0)` — без часового пояса и без долей секунды.

### Задача 6. Выведите выражения с NULL

Выведите каждое выражение из списка в отдельном столбце **col1**, **col2**, **col3**, **col4**.
Затем перепишите последнее выражение так, чтобы оно корректно сравнивало оба значения.
- `1 + null`
- `'ABC' || null`
- `(false and null) or true`
- `null = null`

In [8]:
%%sql
SELECT 1 + null                      AS col1,
       'ABC' || null                 AS col2,
       (false AND null) OR true      AS col3,
       null = null                   AS col4,
       null IS NOT DISTINCT FROM null AS col4_fixed;

,col1,col2,col3,col4,col4_fixed
0,NULL,NULL,true,NULL,true


Строк: 1


**Разбор.** Ответ: `NULL, NULL, true, NULL, true`.

NULL — «неизвестно». Любая арифметика и склейка с неизвестным дают неизвестное.
`false AND null` = `false` (при любом значении null результат ложен), дальше `false OR true` = `true`.
`null = null` — тоже неизвестно, а не `true`. Для сравнения, где NULL считается обычным
значением, — `IS NOT DISTINCT FROM`.

### Задача 7. Переведите метры в футы

В авиации высоту полёта называют в футах. Напишите выражение SQL, которое переводит
высоту 10 100 метров в футы и округляет до целого.

Формула: `футы = метры / 0.3048`

Название столбца — **feet**.

In [9]:
%%sql
SELECT round(10100 / 0.3048) AS feet;

,feet
0,33136


Строк: 1


**Разбор.** Ответ: 33 136 футов.

`0.3048` — точное число типа `numeric`, поэтому деление идёт без потери точности;
`round` без второго аргумента округляет до целого.

### Задача 8. Сравните точность float4 и numeric

Правда ли, что тип `numeric(12,2)` точнее, чем `float4`?

Сравните результат выражения `1 / 100.0`, приведённый к `float4` и к `numeric(12,2)`.

Подсказка: достаточно проверить, равны ли значения.

In [10]:
%%sql
SELECT (1 / 100.0)::float4                              AS as_float4,
       (1 / 100.0)::numeric(12,2)                       AS as_numeric,
       (1 / 100.0)::float4 = (1 / 100.0)::numeric(12,2) AS is_equal,
       (1 / 100.0)::float4::float8                      AS float4_inside;

,as_float4,as_numeric,is_equal,float4_inside
0,0.01,0.01,false,0.009999999776482582


Строк: 1


**Разбор.** Ответ: значения не равны.

`float4` хранит двоичную дробь и не может записать 0,01 точно: внутри лежит
0,009999999776… При выводе число округляется, и выглядит как 0.01, а при сравнении
разница всплывает. `numeric` хранит десятичные цифры и хранит 0,01 точно — поэтому
деньги держат в `numeric`, а `float` оставляют для измерений.

# Часть 2. Создание таблиц и ограничения

Свои таблицы создаём в схеме `public` — отдельно от демобазы, которая лежит в схеме `bookings`, — и пишем имя целиком: `public.имя_таблицы`.

**Памятка**

```sql
CREATE TABLE public.имя (
    столбец  тип  PRIMARY KEY,
    столбец  тип  NOT NULL,
    столбец  тип  CHECK (условие),
    столбец  тип  REFERENCES public.другая_таблица (столбец),
    PRIMARY KEY (столбец_1, столбец_2)   -- составной ключ, вместо PRIMARY KEY у столбца
);

INSERT INTO public.имя (столбец_1, столбец_2)
VALUES (1, 'первая строка'),
       (2, 'вторая строка');

DROP TABLE IF EXISTS public.имя;   -- удалить таблицу, если она есть
```

### Задача 9. Создайте таблицу my_trips и добавьте 3 строки

Создайте в схеме `public` таблицу поездок **my_trips** со столбцами:
- уникальный номер поездки **trip_id**
- город назначения, обязательный столбец **city**
- оценка поездки числом от 1 до 5, необязательный столбец **rating**

**Что нужно сделать:**
- создайте таблицу, задайте подходящие типы данных и первичный ключ;
- добавьте в таблицу три строки.

In [11]:
%%sql
DROP TABLE IF EXISTS public.my_trips;   -- чтобы ячейку можно было перезапустить

CREATE TABLE public.my_trips (
    trip_id integer  PRIMARY KEY,
    city    text     NOT NULL,
    rating  smallint CHECK (rating BETWEEN 1 AND 5)
);

INSERT INTO public.my_trips (trip_id, city, rating)
VALUES (1, 'Санкт-Петербург', 5),
       (2, 'Калининград',     4),
       (3, 'Мурманск',        NULL);

SELECT * FROM public.my_trips;

DROP TABLE
CREATE TABLE
INSERT 0 3


,trip_id,city,rating
0,1,Санкт-Петербург,5
1,2,Калининград,4
2,3,Мурманск,NULL


Строк: 3


**Разбор.** Необязательный столбец — это просто столбец без `NOT NULL`. `CHECK` пропускает NULL:
условие `NULL BETWEEN 1 AND 5` неизвестно, а `CHECK` отклоняет только явное `false`.

Проверьте, что ограничения работают: `INSERT` с `rating = 7` или без `city` падает с ошибкой.

### Задача 10. Создайте авиакомпании и оплаты багажа

Нужна таблица авиакомпаний и таблица, где записываются оплаты сверхнормативного багажа.

- Таблица 1 — **airlines**(airline_id, airline_name, baggage_fee)
- Таблица 2 — **baggage_payments**(airline_id, paid_at, weight_kg)

Столбцы таблицы **airlines**:
- **airline_id** — уникальный номер, целое число
- **airline_name** — название, текст
- **baggage_fee** — плата за килограмм, число с 2 знаками после запятой

Добавьте три авиакомпании:
```
airline_id;airline_name;baggage_fee
1;Альфа Авиа;149.99
2;Бета Эйр;190
3;Гамма Лайн;219.99
```

Столбцы таблицы **baggage_payments**:
- **airline_id** — номер авиакомпании, целое число
- **paid_at** — время оплаты, дата и время
- **weight_kg** — вес сверх нормы в килограммах, целое число

Первичный ключ — составной, из двух полей: airline_id и paid_at.

Настройте первичные ключи в обеих таблицах и внешний ключ. Добавьте 3 оплаты в baggage_payments.

In [12]:
%%sql
DROP TABLE IF EXISTS public.baggage_payments;   -- сначала ссылающуюся таблицу,
DROP TABLE IF EXISTS public.airlines;           -- потом ту, на которую ссылаются

CREATE TABLE public.airlines (
    airline_id   integer       PRIMARY KEY,
    airline_name text          NOT NULL,
    baggage_fee  numeric(8,2)  NOT NULL
);

CREATE TABLE public.baggage_payments (
    airline_id integer   REFERENCES public.airlines (airline_id),
    paid_at    timestamp,
    weight_kg  integer   NOT NULL,
    PRIMARY KEY (airline_id, paid_at)
);

INSERT INTO public.airlines (airline_id, airline_name, baggage_fee)
VALUES (1, 'Альфа Авиа', 149.99),
       (2, 'Бета Эйр',   190),
       (3, 'Гамма Лайн', 219.99);

INSERT INTO public.baggage_payments (airline_id, paid_at, weight_kg)
VALUES (1, '2016-10-13 09:15', 5),
       (1, '2016-10-13 11:40', 3),
       (3, '2016-10-13 12:05', 8);

DROP TABLE
DROP TABLE
CREATE TABLE
CREATE TABLE
INSERT 0 3
INSERT 0 3


**Разбор.** Составной первичный ключ задаётся отдельной строкой `PRIMARY KEY (airline_id, paid_at)`:
два `PRIMARY KEY` у отдельных столбцов PostgreSQL не примет.

Попробуйте получить две ошибки:
- оплата с `airline_id = 4` — нарушение внешнего ключа, такой авиакомпании нет;
- вторая оплата с тем же `airline_id` и `paid_at` — нарушение первичного ключа.

Порядок удаления таблиц обратный порядку создания: таблицу `airlines` не удалить,
пока на неё ссылается `baggage_payments`.

### Задача 11. Прочитайте данные из таблицы baggage_payments

Напишите запрос, который выберет все строки из вашей таблицы baggage_payments по шаблону

`SELECT * FROM название_схемы.название_таблицы;`

In [13]:
%%sql
SELECT * FROM public.baggage_payments;

,airline_id,paid_at,weight_kg
0,1,2016-10-13 09:15:00,5
1,1,2016-10-13 11:40:00,3
2,3,2016-10-13 12:05:00,8


Строк: 3
